# 🎙️ Voice Studio (Colab Edition)

**Script do → AI voiceover MP3 lo** — free, Chatterbox engine ke saath.

### Chalane ka tareeqa
1. **Runtime > Run all** dabao (ya cells ek-ek karke Shift+Enter se chalao)
2. Agar GPU ka error aaye: **Runtime > Change runtime type > GPU (T4)** select karke dobara chalao
3. Step 3 wala cell chalne ke baad neeche tool khul jayega
4. Script paste karo → chaaho to voice sample upload karo (clone) → **Generate Voiceover**
5. MP3 **usi session me download kar lena** — Colab band hote hi files ur jati hain

> Tip: Expressive/dramatic voice ke liye Emotion ~0.7 aur Pacing ~0.3 try karo.

In [ ]:
# Step 1: Zaroori libraries install karo (pehli dafa 3-5 minute lag sakte hain)
!pip install -q chatterbox-tts gradio
print("Install done ✅")

In [ ]:
# Step 2: GPU check + model load (pehli dafa model download hoga, ~1.5 GB)
import torch

if not torch.cuda.is_available():
    raise SystemExit("❌ GPU nahi mila! Menu me jao: Runtime > Change runtime type > Hardware accelerator = GPU (T4) select karo, phir dobara chalao.")

print("GPU:", torch.cuda.get_device_name(0))

from chatterbox.tts import ChatterboxTTS

model = ChatterboxTTS.from_pretrained(device="cuda")
SR = model.sr
print(f"Model ready ✅ (sample rate {SR})")

In [ ]:
# Step 3: Voice Studio app — ye cell chalao, neeche tool khul jayega
import re
import subprocess
import torch
import torchaudio as ta
import gradio as gr

def split_chunks(text, max_chars=300):
    """Lambi script ko sentence boundaries pe chhote chunks me todo (taake voice stable rahe)."""
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, cur = [], ""
    for s in sentences:
        if len(cur) + len(s) + 1 <= max_chars:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            while len(s) > max_chars:  # bohat lamba single sentence -> hard split
                chunks.append(s[:max_chars])
                s = s[max_chars:]
            cur = s
    if cur:
        chunks.append(cur)
    return [c for c in chunks if c]

def generate_voiceover(script, ref_audio, exaggeration, cfg_weight, progress=gr.Progress()):
    chunks = split_chunks(script)
    if not chunks:
        raise gr.Error("Script khali hai — pehle text paste karo.")
    prompt = ref_audio  # clone ke liye sample file, warna None (default voice)
    wavs = []
    silence = torch.zeros(int(SR * 0.35))  # chunk ke darmiyan halka sa waqfa
    for ch in progress.tuple(chunks, desc="Voice ban rahi hai..."):
        wav = model.generate(
            ch,
            audio_prompt_path=prompt,
            exaggeration=exaggeration,
            cfg_weight=cfg_weight,
        )
        wav = wav.squeeze(0) if wav.dim() > 1 else wav
        wavs.append(wav.cpu())
        wavs.append(silence)
    full = torch.cat(wavs[:-1]).unsqueeze(0)  # aakhri silence hata do
    ta.save("/tmp/voiceover.wav", full, SR)
    subprocess.run(
        ["ffmpeg", "-y", "-i", "/tmp/voiceover.wav",
         "-codec:a", "libmp3lame", "-b:a", "192k", "/tmp/voiceover.mp3"],
        check=True, capture_output=True,
    )
    n_min = int(full.shape[1] / SR / 60)
    n_sec = int(full.shape[1] / SR % 60)
    print(f"Done ✅ {len(chunks)} chunks, total ~{n_min}:{n_sec:02d}")
    return "/tmp/voiceover.mp3", "/tmp/voiceover.mp3"

with gr.Blocks(title="Voice Studio") as demo:
    gr.Markdown("# 🎙️ Voice Studio — Script se Voiceover")
    gr.Markdown("Script paste karo, chaaho to apni voice ki 5–10 second ki sample do (clone), Generate dabao, MP3 download kar lo.")
    script = gr.Textbox(label="Script (English)", lines=12,
                        placeholder="Apni poori script yahan paste karo...")
    ref = gr.Audio(label="Voice sample — clone ke liye (optional, 5-10 sec ki saaf recording)",
                   type="filepath")
    with gr.Row():
        exagg = gr.Slider(0.25, 1.0, value=0.5, step=0.05,
                          label="Emotion (Exaggeration) — zyada = zyada expressive")
        cfg = gr.Slider(0.0, 1.0, value=0.5, step=0.05,
                        label="Pacing (Cfg weight) — kam = slow/deliberate")
    btn = gr.Button("🎧 Generate Voiceover", variant="primary")
    out_audio = gr.Audio(label="Sun lo")
    out_file = gr.File(label="MP3 download")
    btn.click(generate_voiceover,
              inputs=[script, ref, exagg, cfg],
              outputs=[out_audio, out_file])

demo.launch()
print("App band ho gayi. Dobara chalane ke liye ye cell phir se chalao.")